# Demo 2 · Event-camera digits (NMNIST) on the chip

Instructor demo. A convolutional spiking network, trained elsewhere and stored as NIR, classifies a handwritten digit recorded with an event camera. One sample takes about one minute on the chip; on Brian2 about one second. Needs `tonic` and `torch` (not part of the participants' requirements); the dataset is downloaded on the first run (about 170 MB).

Adapted from the py-spinnaker2 example "NMNIST with a pre-trained NIR model" (SpiNNcloud, Apache-2.0).

In [ ]:
import matplotlib.pyplot as plt
import nir
import numpy as np
import tonic
from spinnaker2 import brian2_sim, s2_nir

USE_HARDWARE = True  # False: run on Brian2


def backend():
    if USE_HARDWARE:
        from spinnaker2 import hardware

        return hardware.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()

## 1. The data: digits seen by an event camera

NMNIST shows the MNIST digits to an event camera that moves in small saccades. The camera does not record frames, it records events: a pixel reports when its brightness changes. Summed over time, the events show the digit.

In [ ]:
to_frame = tonic.transforms.ToFrame(sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=1e3)  # 1 ms bins
dataset = tonic.datasets.NMNIST("data/nmnist", transform=to_frame, train=False)
frames, label = dataset[1234]  # change the index for another digit; shape (time bins, 2 polarities, 34, 34)
print(f"digit {label}, {frames.shape[0]} time bins, {int(frames.sum())} events")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(frames.sum(axis=(0, 1)), cmap="gray")
axes[0].set_title(f"all events (digit {label})")
for ax, (a, b) in zip(axes[1:], [(0, 100), (100, 200), (200, 300)]):
    ax.imshow(frames[a:b].sum(axis=(0, 1)), cmap="gray")
    ax.set_title(f"events in {a}-{b} ms")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. The network, converted from NIR

In [ ]:
graph = nir.read("data/scnn_mnist.nir")
graph.infer_types()
cfg = s2_nir.ConversionConfig()
cfg.output_record = ["spikes"]
cfg.dt = 0.0001
cfg.conn_delay = 0
cfg.scale_weights = True
cfg.reset = s2_nir.ResetMethod.ZERO
cfg.integrator = s2_nir.IntegratorMethod.FORWARD
net, inp, outp = s2_nir.from_nir(graph, cfg)
for pop in net.populations:
    if pop.name == "6":
        pop.set_max_atoms_per_core(128)  # settings of the original example, to fit the memory of a core
    if pop.name == "10":
        pop.set_max_atoms_per_core(16)
print(f"{len(net.populations)} populations, {sum(p.size for p in net.populations)} neurons")

## 3. Classify one sample on the chip

The predicted digit is the output neuron with the most spikes.

In [ ]:
def classify(frames):
    T, C, H, W = frames.shape
    inp[0].params = {c * H * W + h * W + w: list(np.nonzero(frames[:, c, h, w])[0]) for c in range(C) for h in range(H) for w in range(W)}
    net.reset()
    backend().run(net, T + 1, sys_tick_in_s=2.5e-3)  # 2.5 ms per step, as in the original example
    counts = np.array([len(outp[0].get_spikes().get(i, [])) for i in range(10)])
    return counts


counts = classify(frames)
prediction = int(np.argmax(counts))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(10), counts, color=["crimson" if i == prediction else "0.6" for i in range(10)])
ax.set(xlabel="digit (output neuron)", ylabel="spike count", xticks=range(10),
       title=f"prediction {prediction}, true digit {label} ({'SpiNNaker2' if USE_HARDWARE else 'Brian2'})")
plt.tight_layout()
plt.show()

## 4. What it shows

- Event-camera data is sparse by nature: pixels that see no change send nothing. This is the kind of input neuromorphic hardware is built for (`01_knowledge_base/04_applications.md`, section 4.1).
- The model was trained elsewhere and moved to the chip via NIR, as in hands-on 2. Published work does the same on SpiNNaker2 for gesture recognition with an event camera (Arfa et al. 2025: 94 % accuracy on the chip).
- One sample per run takes about one minute here because every run boots the board; a deployed system would stream the events into a running network.